In [55]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.decoders import BPEDecoder
from tokenizers.pre_tokenizers import Whitespace

from torch.utils.data import Dataset
from torch import nn
import torch
from self_attention import MultiHeadSelfAttention
import math

import pandas as pd


# Open and inspect data

In [56]:
df = pd.read_csv("data/archive/train.csv")
val_df = pd.read_csv("data/archive/validation.csv")

In [57]:


print(df.iloc[9]["dialog"])



["How are Zina's new programmers working out ? "
 " I hate to admit it , but they're good . And fast . The Filipino kid is a genius . "
 " So you'll make the Stars.com deadline , and have us up and running next week ? "
 " It'll be close , but we'll make it . "
 " Good . After Stars.com starts paying us , we won't need Vikam's cash anymore . "
 " And if we don't need them , we won't need Zina , either . "]


# concatenate all rows in text

In [58]:
import ast

df["raw_text"] = (
    df["dialog"]
    .str.replace(r"[\r\n]+|\s['\"]|['\"]\s", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)
val_df["raw_text"] = (
    val_df["dialog"]
    .str.replace(r"[\r\n]+|\s['\"]|['\"]\s", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

print(df.iloc[1]["raw_text"])



['Can you do push-ups ? Of course I can . It's a piece of cake ! Believe it or not , I can do 30 push-ups a minute . Really ? I think that's impossible ! You mean 30 push-ups ? Yeah ! It's easy . If you do exercise everyday , you can make it , too . ]


Start by tokenizing the data using BPE

In [59]:

# Remove empty values
texts = (
    df["raw_text"]
    .dropna()
    .astype(str)
)

# Create tokenizer
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()

trainer = BpeTrainer(
    vocab_size=5000,
    special_tokens=[
        "[PAD]",
        "[UNK]",
        "[CLS]",
        "[SEP]",
        "[MASK]"
    ]
)

# Train directly from pandas series
tokenizer.train_from_iterator(
    texts,
    trainer=trainer
)

# Save
tokenizer.save("email_tokenizer.json")


In [60]:


class TextDataset(Dataset):
    def __init__(self, token_ids, context_len=64):
        self.data = token_ids
        self.ctx = context_len

    def __len__(self):
        return len(self.data) - self.ctx

    def __getitem__(self, i):
        x = self.data[i : i + self.ctx]
        y = self.data[i + 1 : i + self.ctx + 1]  # shifted by 1
        return torch.tensor(x), torch.tensor(y)

In [61]:
class WordPredictor(nn.Module):
    def __init__(self, vocab_size, d_model=256, nhead=4, num_layers=4):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos   = nn.Embedding(512, d_model)
        encoder_layer = nn.TransformerEncoderLayer(d_model, nhead, 
                            dim_feedforward=1024, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers)
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        positions = torch.arange(x.size(1), device=x.device)
        out = self.embed(x) + self.pos(positions)
        mask = nn.Transformer.generate_square_subsequent_mask(x.size(1), device=x.device)
        out = self.transformer(out, mask=mask, is_causal=True)
        return self.head(out)

In [62]:
class TransformerLM(nn.Module):
    def __init__(self, vocab_size, d_model=256, nhead=8, num_layers=4, 
                 dim_feedforward=1024, max_seq_len=512, dropout=0.1):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)
        
        decoder_layer = nn.TransformerDecoderLayer(d_model, nhead, dim_feedforward, dropout, batch_first=True)
        self.transformer = nn.TransformerDecoder(decoder_layer, num_layers)
        
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)
        self.d_model = d_model

    def forward(self, x):
        B, T = x.shape
        positions = torch.arange(T, device=x.device).unsqueeze(0)
        
        x = self.token_emb(x) * math.sqrt(self.d_model) + self.pos_emb(positions)
        
        # Causal mask — each token can only attend to previous tokens
        causal_mask = nn.Transformer.generate_square_subsequent_mask(T, device=x.device)
        
        # Decoder-only (GPT-style): memory = x, apply causal mask
        out = self.transformer(x, x, tgt_mask=causal_mask, memory_mask=causal_mask)
        out = self.norm(out)
        return self.head(out)  # (B, T, vocab_size)

In [63]:
from dataclasses import dataclass

@dataclass
class Config :
    vocab_size: int = 5000  # This number should agree with the tokenizer
    number_of_transformer_blocks: int = 4
    number_of_attention_heads: int = 4
    vector_dim: int = 256
    block_size: int = 512
    dropout_prob: float = 0.1
    batch_size: int = 8
    learning_rate: float = 0.0005
    weight_decay: float = 0.000001
    no_of_epochs: int = 1

class PositionwiseFFN(nn.Module):
    """
    The position-wise FFN that follows after the self-attention computation.
    Vectors are projected to 4x the dimensionality and then projected down
    again after relu application.
    """

    def __init__(self, vector_dim, dropout_prob) :
        super().__init__()
        self.fc1 = nn.Linear(vector_dim, 4*vector_dim, bias=True)
        self.fc2 = nn.Linear(4*vector_dim, vector_dim, bias=True)
        self.dropout = nn.Dropout(dropout_prob)

    def forward(self, x):
        return self.fc2(self.dropout(torch.relu(self.fc1(x))))

class Block(nn.Module):
    """
    Transformer encoder block.

    This version differs from the original version in  [Vaswani et al. NeurIPS 2017],
    and applies the LayerNorm before the self-attention, and before the FFN, as this
    has proved to be beneficial (see [Nguyen and Salazar 2019]).
    """

    def __init__(self, vector_dim, n_heads, block_size, dropout_prob):
        super().__init__()
        att_dim = vector_dim // n_heads
        self.attn = MultiHeadSelfAttention(vector_dim, n_heads, block_size, is_causal=True)
        self.ffn = PositionwiseFFN(vector_dim, dropout_prob)
        self.dropout = nn.Dropout(dropout_prob)
        self.ln1 = nn.LayerNorm(vector_dim)
        self.ln2 = nn.LayerNorm(vector_dim)

    def forward(self, x):
        x1 = self.ln1(x)
        x2 = x + self.dropout(self.attn(x1))
        x3 = self.ln2(x2)
        x4 = x2 + self.dropout(self.ffn(x3))
        return x4

class TinyStoriesLM(nn.Module):

    def __init__(self, config):
        super(TinyStoriesLM, self).__init__()
        self.config = config
        self.embed =  nn.Embedding(config.vocab_size, config.vector_dim)
        self.positional = nn.Parameter(torch.randn(1, config.block_size, config.vector_dim))
        modules = [Block(config.vector_dim,\
                         config.number_of_attention_heads,\
                         config.block_size,\
                         config.dropout_prob) for _ in range(config.number_of_transformer_blocks)]
        self.transformers = nn.ModuleList(modules)
        self.final = nn.Linear(config.vector_dim, config.vocab_size)

    def forward(self, x):

        # YOUR CODE HERE
        B, T = x.shape
        token_embed = self.embed(x) # (B, T, C)
        pos_embed = self.positional[:, :T, :] # (1, T, C)
        x = token_embed + pos_embed # (B, T, C)

        for block in self.transformers:
            x = block(x)

        logits = self.final(x)
        
        
        return logits

# Tokenizing training data

In [64]:
print("importing additional libraries for training")

from torch.utils.data import DataLoader
from torch.optim import Adam
import json
import os

def tokenize_and_save(tokenizer, df, filename):
    texts = df["raw_text"].dropna().astype(str).tolist()
    encoded_batch = tokenizer.encode_batch(texts)
    print("Batch encoding complete")

    token_ids = []
    for i, encoded in enumerate(encoded_batch):
        token_ids.extend(encoded.ids)



    print(f"Total tokens: {len(token_ids)}")

    # Save for next time
    print("Saving token IDs...")
    with open(filename, "w") as f:
        json.dump(token_ids, f)

tokenize_and_save(tokenizer, df, "token_ids.json")
tokenize_and_save(tokenizer, val_df, "val_token_ids.json")



importing additional libraries for training
Batch encoding complete
Total tokens: 1379350
Saving token IDs...
Batch encoding complete
Total tokens: 126912
Saving token IDs...


In [65]:
from torch.utils.data import Subset

# Create dataset and dataloader
with open("token_ids.json", "r") as f:
    training_ids = json.load(f)
with open("val_token_ids.json", "r") as f:
    val_ids = json.load(f)

dataset = TextDataset(training_ids, context_len=64)
val_dataset = TextDataset(val_ids, context_len=64)
print(f"training length: {len(dataset)}, validation length: {len(val_dataset)}")

dataloader = DataLoader(dataset, batch_size=32, shuffle=True)
val_dataloader = DataLoader(Subset(val_dataset, range(1000)), batch_size=32)

training length: 1379286, validation length: 126848


In [66]:
def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)
            output = model(x)
            loss = criterion(output.view(-1, output.size(-1)), y.view(-1))
            total_loss += loss.item()
    return total_loss / len(dataloader)

def train(model, dataloader, val_dataloader, epochs=1, lr=1e-3, verbose=True):
    device = next(model.parameters()).device
    optimizer = Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    
    for epoch in range(epochs):
        total_loss = 0
        for i, (x, y) in enumerate(dataloader):
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            output = model(x)  # (B, T, vocab_size)
            loss = criterion(output.view(-1, output.size(-1)), y.view(-1))
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            if verbose and (i + 1) % 100 == 0:
                avg_loss = total_loss / 100
                print(f"Epoch {epoch+1}, Step {i+1}/{len(dataloader)}, Loss: {avg_loss:.4f}")
                total_loss = 0
            if verbose and (i + 1) % 1000 == 0:
                val_loss = evaluate(model, val_dataloader, criterion, device)
                model.train()
                print(f"Epoch {epoch+1}, Step {i+1}/{len(dataloader)}, Validation Loss: {val_loss:.4f}")

In [94]:
# Initialize model
print("initializing model...")
vocab_size = tokenizer.get_vocab_size()
device = torch.device("mps" if torch.mps.is_available() else "cpu")
print(f"Using device: {device}")

# model = WordPredictor(vocab_size, d_model=256, nhead=4, num_layers=4)
# model = TransformerLM(vocab_size=vocab_size, d_model=256, nhead=8, num_layers=4)

config = Config()
# model = TinyStoriesLM(config)
model = WordPredictor(vocab_size, d_model=256, nhead=4, num_layers=4)
# model.load_state_dict(torch.load("word_predictor_lower.pth", map_location=device))
model.to(device)
print(f"tokenizer vocab size: {vocab_size} Config vocab size: {config.vocab_size}")

model.train()
train(model, dataloader, val_dataloader, epochs=1, verbose=True)

print("Training complete!")

initializing model...
Using device: mps
tokenizer vocab size: 5000 Config vocab size: 5000
Epoch 1, Step 100/43103, Loss: 5.8123
Epoch 1, Step 200/43103, Loss: 4.9534
Epoch 1, Step 300/43103, Loss: 4.6473
Epoch 1, Step 400/43103, Loss: 4.4855
Epoch 1, Step 500/43103, Loss: 4.3528
Epoch 1, Step 600/43103, Loss: 4.2878
Epoch 1, Step 700/43103, Loss: 4.1992
Epoch 1, Step 800/43103, Loss: 4.1383
Epoch 1, Step 900/43103, Loss: 4.0705
Epoch 1, Step 1000/43103, Loss: 3.9907
Epoch 1, Step 1000/43103, Validation Loss: 3.8940
Epoch 1, Step 1100/43103, Loss: 3.9471
Epoch 1, Step 1200/43103, Loss: 3.9033
Epoch 1, Step 1300/43103, Loss: 3.8729
Epoch 1, Step 1400/43103, Loss: 3.8567
Epoch 1, Step 1500/43103, Loss: 3.8151
Epoch 1, Step 1600/43103, Loss: 3.7851
Epoch 1, Step 1700/43103, Loss: 3.7527
Epoch 1, Step 1800/43103, Loss: 3.7076
Epoch 1, Step 1900/43103, Loss: 3.6962
Epoch 1, Step 2000/43103, Loss: 3.6767
Epoch 1, Step 2000/43103, Validation Loss: 3.6250
Epoch 1, Step 2100/43103, Loss: 3.6445

In [95]:
# save the model
torch.save(model.state_dict(), "word_predictor_conv.pth")

In [101]:
# Load model for prediction
model = TinyStoriesLM(config)  # Make sure to initialize the same architecture
model.load_state_dict(torch.load("tinyStories_model_conv.pth", map_location=device))
model.to(device)

TinyStoriesLM(
  (embed): Embedding(5000, 256)
  (transformers): ModuleList(
    (0-3): 4 x Block(
      (attn): MultiHeadSelfAttention(
        (wq): Linear(in_features=256, out_features=256, bias=False)
        (wk): Linear(in_features=256, out_features=256, bias=False)
        (wv): Linear(in_features=256, out_features=256, bias=False)
        (wo): Linear(in_features=256, out_features=256, bias=False)
      )
      (ffn): PositionwiseFFN(
        (fc1): Linear(in_features=256, out_features=1024, bias=True)
        (fc2): Linear(in_features=1024, out_features=256, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (dropout): Dropout(p=0.1, inplace=False)
      (ln1): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
      (ln2): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
    )
  )
  (final): Linear(in_features=256, out_features=5000, bias=True)
)

In [147]:
def predict_next_word(model, tokenizer, prompt, prefix, context_len=128, top_k = 100, device=None):
    model.eval()
    with torch.no_grad():
        encoded = tokenizer.encode(prompt)
        token_ids = encoded.ids[-context_len:]  # take last context_len tokens
        x = torch.tensor(token_ids).unsqueeze(0).to(device)  # shape (1, context_len)
        
        logits = model(x)  # shape (1, context_len, vocab_size)
        last_logits = logits[0, -1]  # shape (vocab_size,)
        top_k_indices = torch.topk(last_logits, top_k).indices.cpu().numpy()
        predicted_words = [tokenizer.decode([idx]) for idx in top_k_indices]
        predicted_words = [w for w in predicted_words if w.startswith(prefix)]

        return predicted_words


prompt = "I am"
prefix = "w"
predictions = predict_next_word(model, tokenizer, prompt, prefix, context_len=128, top_k=1000, device=device)
print(predictions)

['worried', 'working', 'with', 'why', 'waiting', 'wi', 'worrying', 'what', 'welcome', 'watching', 'wearing', 'w', 'wondering', 'willing', 'withdraw', 'well', 'white', 'wife', 'where', 'work', 'writing', 'wrong', 'whatever', 'worth', 'walking', 'watch', 'we', 'world', 'wal', 'weak', 'wanted', 'warranty', 'wide', 'wn']


# Evalutation